In [1]:
import polars as pl

pl.Config.set_tbl_rows(24)
pl.Config.save()  # saves to ~/.config/polars/config.json

'{"environment":{"POLARS_ENGINE_AFFINITY":null,"POLARS_FMT_MAX_COLS":null,"POLARS_FMT_MAX_ROWS":"24","POLARS_FMT_NUM_DECIMAL":null,"POLARS_FMT_NUM_GROUP_SEPARATOR":null,"POLARS_FMT_NUM_LEN":null,"POLARS_FMT_STR_LEN":null,"POLARS_FMT_TABLE_CELL_ALIGNMENT":null,"POLARS_FMT_TABLE_CELL_LIST_LEN":null,"POLARS_FMT_TABLE_CELL_NUMERIC_ALIGNMENT":null,"POLARS_FMT_TABLE_DATAFRAME_SHAPE_BELOW":null,"POLARS_FMT_TABLE_FORMATTING":null,"POLARS_FMT_TABLE_HIDE_COLUMN_DATA_TYPES":null,"POLARS_FMT_TABLE_HIDE_COLUMN_NAMES":null,"POLARS_FMT_TABLE_HIDE_COLUMN_SEPARATOR":null,"POLARS_FMT_TABLE_HIDE_DATAFRAME_SHAPE_INFORMATION":null,"POLARS_FMT_TABLE_INLINE_COLUMN_DATA_TYPE":null,"POLARS_FMT_TABLE_ROUNDED_CORNERS":null,"POLARS_MAX_EXPR_DEPTH":null,"POLARS_STREAMING_CHUNK_SIZE":null,"POLARS_TABLE_WIDTH":null,"POLARS_VERBOSE":null,"POLARS_WARN_UNSTABLE":null},"direct":{"set_fmt_float":"mixed","set_float_precision":null,"set_thousands_separator":"","set_decimal_separator":".","set_trim_decimal_zeros":false}}'

In [2]:
lf = pl.scan_parquet(
    "../data/raw/yellow_tripdata_2025-01.parquet"
)  # lazy scan - loads on collect()
df = lf.collect()
df.columns = [c.lower() for c in df.columns]
df = df.rename(
    {
        "ratecodeid": "rate_code_id",
        "vendorid": "vendor_id",
        "pulocationid": "pickup_location_id",
        "dolocationid": "dropoff_location_id",
    }
)

In [3]:
zones = pl.read_csv("../data/raw/taxi_zone_lookup.csv")
df = df.join(
    zones.rename({"LocationID": "pickup_location_id"}),
    on="pickup_location_id",
    how="left",
)

In [4]:
zones.filter(pl.col("Zone").str.contains("Airport"))

LocationID,Borough,Zone,service_zone
i64,str,str,str
1,"""EWR""","""Newark Airport""","""EWR"""
132,"""Queens""","""JFK Airport""","""Airports"""
138,"""Queens""","""LaGuardia Airport""","""Airports"""


In [5]:
# Build model columns
store_and_fwd_bin = {
    "N": 0,
    "Y": 1,
}

df = df.with_columns(
    # Create duration_sec column from dropoff and pickup times
    (
        (pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime")).dt.total_seconds() / 60
    ).alias("duration_min"),
    # Convert Y/N flagging to 1-hot encoding
    (
        pl.col("store_and_fwd_flag")
        .replace_strict(store_and_fwd_bin, default=None)
        .alias("store_and_fwd_flag")
    ),
)

In [6]:
# %%script echo skipping
# print("This code will be completely ignored.")
# EDA ONLY — drop before modeling
# Build readability columns
vendor_id_map = {
    1: "Creative Mobile Technologies LLC",
    2: "Curb Mobility, LLC",
    6: "Myle Technologies Inc",
    7: "Helix",
}

payment_type_map = {
    0: "flex_fare_trip",
    1: "credit_card",
    2: "cash",
    3: "no_charge",
    4: "dispute",
    5: "unknown",
    6: "voided_trip",
}


rate_code_map = {
    1: "standard_rate",
    2: "jfk",
    3: "newark",
    4: "nassau_or_winchester",
    5: "negotiated_fare",
    6: "group_ride",
    99: "null_or_unknown",
}

df = df.with_columns(
    # Create a new column with readable payment type
    (
        pl.col("payment_type")
        .replace_strict(payment_type_map, default=None)
        .alias("payment_type_str")
    ),
    # Create a new column with readable rate code
    (pl.col("rate_code_id").replace_strict(rate_code_map, default=None).alias("rate_code_str")),
    # Create a new column with readable vendor ID
    (pl.col("vendor_id").replace_strict(vendor_id_map, default=None).alias("vendor_id_str")),
)

In [7]:
df.schema

Schema([('vendor_id', Int32),
        ('tpep_pickup_datetime', Datetime(time_unit='us', time_zone=None)),
        ('tpep_dropoff_datetime', Datetime(time_unit='us', time_zone=None)),
        ('passenger_count', Int64),
        ('trip_distance', Float64),
        ('rate_code_id', Int64),
        ('store_and_fwd_flag', Int64),
        ('pickup_location_id', Int32),
        ('dropoff_location_id', Int32),
        ('payment_type', Int64),
        ('fare_amount', Float64),
        ('extra', Float64),
        ('mta_tax', Float64),
        ('tip_amount', Float64),
        ('tolls_amount', Float64),
        ('improvement_surcharge', Float64),
        ('total_amount', Float64),
        ('congestion_surcharge', Float64),
        ('airport_fee', Float64),
        ('cbd_congestion_fee', Float64),
        ('Borough', String),
        ('Zone', String),
        ('service_zone', String),
        ('duration_min', Float64),
        ('payment_type_str', String),
        ('rate_code_str', String),
       

In [8]:
# Filter out rows where the duration is less than or equal to 0
df.filter(pl.col("duration_min") > 0)

vendor_id,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,Borough,Zone,service_zone,duration_min,payment_type_str,rate_code_str,vendor_id_str
i32,datetime[μs],datetime[μs],i64,f64,i64,i64,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,f64,str,str,str
1,2025-01-01 00:18:38,2025-01-01 00:26:59,1,1.6,1,0,229,237,1,10.0,3.5,0.5,3.0,0.0,1.0,18.0,2.5,0.0,0.0,"""Manhattan""","""Sutton Place/Turtle Bay North""","""Yellow Zone""",8.35,"""credit_card""","""standard_rate""","""Creative Mobile Technologies L…"
1,2025-01-01 00:32:40,2025-01-01 00:35:13,1,0.5,1,0,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0,"""Manhattan""","""Upper East Side North""","""Yellow Zone""",2.55,"""credit_card""","""standard_rate""","""Creative Mobile Technologies L…"
1,2025-01-01 00:44:04,2025-01-01 00:46:01,1,0.6,1,0,141,141,1,5.1,3.5,0.5,2.0,0.0,1.0,12.1,2.5,0.0,0.0,"""Manhattan""","""Lenox Hill West""","""Yellow Zone""",1.95,"""credit_card""","""standard_rate""","""Creative Mobile Technologies L…"
2,2025-01-01 00:14:27,2025-01-01 00:20:01,3,0.52,1,0,244,244,2,7.2,1.0,0.5,0.0,0.0,1.0,9.7,0.0,0.0,0.0,"""Manhattan""","""Washington Heights South""","""Boro Zone""",5.566667,"""cash""","""standard_rate""","""Curb Mobility, LLC"""
2,2025-01-01 00:21:34,2025-01-01 00:25:06,3,0.66,1,0,244,116,2,5.8,1.0,0.5,0.0,0.0,1.0,8.3,0.0,0.0,0.0,"""Manhattan""","""Washington Heights South""","""Boro Zone""",3.533333,"""cash""","""standard_rate""","""Curb Mobility, LLC"""
2,2025-01-01 00:48:24,2025-01-01 01:08:26,2,2.63,1,0,239,68,2,19.1,1.0,0.5,0.0,0.0,1.0,24.1,2.5,0.0,0.0,"""Manhattan""","""Upper West Side South""","""Yellow Zone""",20.033333,"""cash""","""standard_rate""","""Curb Mobility, LLC"""
1,2025-01-01 00:14:47,2025-01-01 00:16:15,0,0.4,1,0,170,170,1,4.4,3.5,0.5,2.35,0.0,1.0,11.75,2.5,0.0,0.0,"""Manhattan""","""Murray Hill""","""Yellow Zone""",1.466667,"""credit_card""","""standard_rate""","""Creative Mobile Technologies L…"
1,2025-01-01 00:39:27,2025-01-01 00:51:51,0,1.6,1,0,234,148,1,12.1,3.5,0.5,2.0,0.0,1.0,19.1,2.5,0.0,0.0,"""Manhattan""","""Union Sq""","""Yellow Zone""",12.4,"""credit_card""","""standard_rate""","""Creative Mobile Technologies L…"
1,2025-01-01 00:53:43,2025-01-01 01:13:23,0,2.8,1,0,148,170,1,19.1,3.5,0.5,3.0,0.0,1.0,27.1,2.5,0.0,0.0,"""Manhattan""","""Lower East Side""","""Yellow Zone""",19.666667,"""credit_card""","""standard_rate""","""Creative Mobile Technologies L…"


In [9]:
df.describe()

statistic,vendor_id,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,Borough,Zone,service_zone,duration_min,payment_type_str,rate_code_str,vendor_id_str
str,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,f64,str,str,str
"""count""",3.475226e6,"""3475226""","""3475226""",2.935077e6,3.475226e6,2.935077e6,2.935077e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,2.935077e6,2.935077e6,3.475226e6,"""3475226""","""3475226""","""3475226""",3.475226e6,"""3475226""","""2935077""","""3475226"""
"""null_count""",0.0,"""0""","""0""",540149.0,0.0,540149.0,540149.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,540149.0,540149.0,0.0,"""0""","""0""","""0""",0.0,"""0""","""540149""","""0"""
"""mean""",1.785428,"""2025-01-17 11:02:55.910964""","""2025-01-17 11:17:56.997901""",1.297859,5.855126,2.482535,0.002605,165.191576,164.125177,1.036623,17.081803,1.317737,0.478099,2.959813,0.449308,0.954795,25.611292,2.225237,0.123911,0.483409,null,null,null,15.018116,null,null,null
"""std""",0.426328,null,null,0.75075,564.6016,11.632772,0.050973,64.529483,69.401686,0.701333,463.472918,1.861509,0.137462,3.779681,2.002582,0.278194,463.658478,0.903993,0.472509,0.361931,null,null,null,38.713582,null,null,null
"""min""",1.0,"""2024-12-31 20:47:55""","""2024-12-18 07:52:40""",0.0,0.0,1.0,0.0,1.0,1.0,0.0,-900.0,-7.5,-0.5,-86.0,-126.94,-1.0,-901.0,-2.5,-1.75,-0.75,"""Bronx""","""Allerton/Pelham Gardens""","""Airports""",-51472.316667,"""cash""","""group_ride""","""Creative Mobile Technologies L…"
"""25%""",2.0,"""2025-01-10 07:59:01""","""2025-01-10 08:15:29""",1.0,0.98,1.0,0.0,132.0,113.0,1.0,8.6,0.0,0.5,0.0,0.0,1.0,15.2,2.5,0.0,0.0,null,null,null,7.283333,null,null,null
"""50%""",2.0,"""2025-01-17 15:41:34""","""2025-01-17 15:59:34""",1.0,1.67,1.0,0.0,162.0,162.0,1.0,12.11,0.0,0.5,2.45,0.0,1.0,19.95,2.5,0.0,0.75,null,null,null,11.7,null,null,null
"""75%""",2.0,"""2025-01-24 19:34:06""","""2025-01-24 19:48:31""",1.0,3.1,1.0,0.0,234.0,234.0,1.0,19.5,2.5,0.5,3.93,0.0,1.0,27.78,2.5,0.0,0.75,null,null,null,18.333333,null,null,null
"""max""",7.0,"""2025-02-01 00:00:44""","""2025-02-01 23:44:11""",9.0,276423.57,99.0,1.0,265.0,265.0,5.0,863372.12,15.0,10.5,400.0,170.94,1.0,863380.37,2.5,6.75,0.75,"""Unknown""","""Yorkville West""","""Yellow Zone""",5626.316667,"""unknown""","""standard_rate""","""Myle Technologies Inc"""


In [10]:
df.filter(pl.col("duration_min") < 0).sort("duration_min")

vendor_id,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,Borough,Zone,service_zone,duration_min,payment_type_str,rate_code_str,vendor_id_str
i32,datetime[μs],datetime[μs],i64,f64,i64,i64,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,f64,str,str,str
1,2025-01-23 01:44:59,2024-12-18 07:52:40,0,3.1,1,0,162,238,1,21.9,2.5,0.5,4.5,0.0,1.0,30.4,2.5,0.0,0.0,"""Manhattan""","""Midtown East""","""Yellow Zone""",-51472.316667,"""credit_card""","""standard_rate""","""Creative Mobile Technologies L…"
1,2025-01-02 12:26:00,2025-01-02 11:29:58,1,9.0,99,0,88,210,1,45.5,0.0,0.5,0.0,0.0,0.0,46.0,0.0,0.0,0.0,"""Manhattan""","""Financial District South""","""Yellow Zone""",-56.033333,"""credit_card""","""null_or_unknown""","""Creative Mobile Technologies L…"
1,2025-01-06 16:00:00,2025-01-06 15:05:30,1,3.8,99,0,208,254,1,24.5,0.0,0.5,0.0,0.0,0.0,25.0,0.0,0.0,0.0,"""Bronx""","""Schuylerville/Edgewater Park""","""Boro Zone""",-54.5,"""credit_card""","""null_or_unknown""","""Creative Mobile Technologies L…"
1,2025-01-23 12:30:00,2025-01-23 11:44:59,1,3.9,99,0,236,116,1,27.5,0.0,0.5,0.0,0.0,0.0,28.0,0.0,0.0,0.0,"""Manhattan""","""Upper East Side North""","""Yellow Zone""",-45.016667,"""credit_card""","""null_or_unknown""","""Creative Mobile Technologies L…"
1,2025-01-29 14:00:00,2025-01-29 13:30:15,1,1.5,99,0,72,222,1,19.5,0.0,0.5,0.0,0.0,0.0,20.0,0.0,0.0,0.0,"""Brooklyn""","""East Flatbush/Remsen Village""","""Boro Zone""",-29.75,"""credit_card""","""null_or_unknown""","""Creative Mobile Technologies L…"
1,2025-01-15 15:00:00,2025-01-15 14:42:48,1,1.0,99,0,107,4,1,18.5,0.0,0.5,0.0,0.0,0.0,19.0,0.0,0.0,0.0,"""Manhattan""","""Gramercy""","""Yellow Zone""",-17.2,"""credit_card""","""null_or_unknown""","""Creative Mobile Technologies L…"
6,2025-01-17 13:01:56,2025-01-17 13:01:00,null,1.05,null,null,7,7,0,2.9,0.0,0.5,0.0,0.0,0.3,16.0,null,null,0.0,"""Queens""","""Astoria""","""Boro Zone""",-0.933333,"""flex_fare_trip""",null,"""Myle Technologies Inc"""
6,2025-01-13 00:01:59,2025-01-13 00:01:05,null,3.25,null,null,37,102,0,2.9,0.0,0.5,0.0,0.0,0.3,17.88,null,null,0.0,"""Brooklyn""","""Bushwick South""","""Boro Zone""",-0.9,"""flex_fare_trip""",null,"""Myle Technologies Inc"""
6,2025-01-13 13:01:51,2025-01-13 13:01:00,null,2.62,null,null,263,170,0,2.9,0.0,0.5,0.0,0.0,0.3,27.6,null,null,0.0,"""Manhattan""","""Yorkville West""","""Yellow Zone""",-0.85,"""flex_fare_trip""",null,"""Myle Technologies Inc"""


In [11]:
# Zero column analysis -- Clean up output, i.e. pivot column names to rows with one column "count"
cols = [
    "passenger_count",
    "trip_distance",
    "duration_min",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
    "cbd_congestion_fee",
]
pl.concat(
    [df.select(pl.lit(c).alias("column"), (pl.col(c) == 0).sum().alias("zero_count")) for c in cols]
)

column,zero_count
str,u32
"""passenger_count""",24656
"""trip_distance""",90893
"""duration_min""",1927
"""fare_amount""",1398
"""extra""",1764424
"""mta_tax""",38170
"""tip_amount""",1118008
"""tolls_amount""",3259590
"""improvement_surcharge""",37694


In [12]:
# Tail analysis
cols = ["trip_distance", "passenger_count", "duration_min"]
cast_df = df.with_columns(pl.col(cols).cast(pl.Float64))
pl.concat(
    [
        cast_df.select(
            pl.lit(c).alias("column"),
            pl.col(c).min().alias("min"),
            pl.col(c).quantile(0.001).alias("p0.1"),
            pl.col(c).quantile(0.999).alias("p99.9"),
            pl.col(c).max().alias("max"),
        )
        for c in cols
    ]
)

column,min,p0.1,p99.9,max
str,f64,f64,f64,f64
"""trip_distance""",0.0,0.0,28.86,276423.57
"""passenger_count""",0.0,0.0,6.0,9.0
"""duration_min""",-51472.316667,0.066667,98.266667,5626.316667


In [13]:
df.group_by(pl.col("tpep_pickup_datetime").dt.hour().alias("hour")).agg(
    pl.col("duration_min").mean().alias("avg_duration")
).sort("hour")

hour,avg_duration
i8,f64
0,13.785256
1,12.06919
2,12.763833
3,12.500513
4,14.155369
5,15.998068
6,15.67766
7,15.709326
8,15.574044


In [14]:
df["payment_type"].value_counts()

payment_type,count
i64,u32
0,540149
4,76481
2,390429
5,1
1,2444393
3,23773


In [15]:
zone_counts = df["Zone"].value_counts()

# EDA Notes

### Fields Known at Pickup:

- **VendorID** - I assume this has to be the taxi proprietor
- RatecodeID
- **tpep_pickup_datetime** - goes without saying
- **passenger_count** - again, you should know how many people are getting in the car
- **store_and_fwd_flag**(?) - not sure about this one
- **PULocationID** - again, you should know where you are
- **DOLocationID** - I think there's an argument to be made about this one. You have to tell the cabby where you want to go as soon as you get in the car, so they'd have to know the drop-off location at pickup, right? It is the definition itself that is making me pause: "TLC Taxi Zone in which the taximeter was disengaged."

### Pickup/Dropoff

- Earliest pickup datetime is after the earliest dropoff datetime
  - Going to need to remove records where:
    - tpep_dropoff_datetime <= tpep_pickup_datetime

### Zero Values

- **Trip Distance:** 90,893 zero distance records
- **Passenger Count:** 24,656 zero passenger records
- **Fare Amount:** 1,398 zero passenger records (May or may not be legitimate)
  - Small population suggests that this is an error rather than legit zeros.
- **Total Amount:** 559 zero passenger records (May or may not be legitimate)
  - Small population suggests that this is an error rather than legit zeros.
  - _NOTE:_ Zero total amount population is smaller than zero fare amount population./nWarrants further investigation as this could be either:
    - The ride was actually free and the individual provided a tip or there was a categorical surcharge present
    - 559 of the 1,398 zero fare amount records should have a larger value present and the remainder were legitimate zero fare rides
- **Trip Duration:** Only 1,927 zero minute duration rides is vast difference from the 90,893 zero trip distance records
  - Could be cab starts meter after stopping for pickup and then ride is cancelled with time charged?
    - I don't live in NY, so I'm not sure how this works.

### Tail Analysis

- Trip distance has massive max error tail
  - p99.9 is only 28.86, so 276423.57 is both, per the dataset and mathematically, impossible
- Passenger count has only 18 records > 6.0 (four 7.0 records, eleven 8.0 records, and three 9.0 records)
  - Given there are millions of records, it seems wiser to just cut these out, although it is technically possible that there may have been babies present with parents or people sitting on laps illegally, just saying.
- Any negative trip duration/trip distance values can be removed from the dataset.

### Cross-Column Relationship Issues

- **Trip Duration x Trip Distance**
  - Potential problem in terms of its error tail. It's hard to say where the errors are and where some of them are just truly long trips when cross referenced with **Trip Distance.** There is a 91 mile trip where the meter ran for 5000+ minutes. Given that that is 3.5 days of usage, I doubt the time is entirely legitimate, but it is a long haul trip for sure. There are also hundreds where it ran for 300+ minutes on a 15-20 mile trip. So it's difficult to pin down where to actually remove meter errors and which are legitimate fares/trip durations.
  - Duration should have a minimum cutoff as well in relation to distance of some sort. Trips less than 1 minute seem ridiculous logically, but also, if something has a long distance then there needs to be a logical amount of trip duration for that distance.
  - _NOTE:_ Perhaps get trimmed, weighted average for time per mile travelled to determine a reasonable min/max for the cross-column relationship and remove anything that falls outside of that. I assume there are going to be accuracy issues as part of this. Could also create some sort of likelihood value of the mile/minute value and then filter based on that?
-
